In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Bronze_Ingestion").getOrCreate()
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import logging

In [0]:
source_path = "/Volumes/rwd/bronze/myvolume/bronze/staging/encounter/"
checkpoint_path = "/Volumes/rwd/bronze/myvolume/bronze/encounter_raw/checkpoint/"
schema_location = "/Volumes/rwd/bronze/myvolume/bronze/encounter_raw/schema/"
BRONZE_TABLE = "rwd.bronze.encounter"


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType

# Define Encounter schema
bronze_schema = StructType([
    StructField("Encounter_ID", StringType(), nullable=False),
    StructField("Patient_ID", StringType(), nullable=False),
    StructField("Cancer_Disease_Status", StringType(), nullable=False),
    StructField("Last_Update", DateType(), nullable=False),
    StructField("On_Treatment", StringType(), nullable=False)
])

In [0]:
def setup_tables():

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
            Encounter_ID              STRING       ,
            Patient_ID                 STRING,
            Cancer_Disease_Status       STRING,
            Last_Update                 DATE,
            On_Treatment                STRING,
            _source_file            STRING,
            _source_file_mtime      TIMESTAMP,
            _load_datetime          TIMESTAMP,
            _batch_id               STRING
        )
        USING DELTA
        TBLPROPERTIES (
            'delta.enableChangeDataFeed'       = 'true',
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """)


In [0]:
# Call setup_tables to create the Bronze table
#setup_tables()

In [0]:
def ingest_bronze():

    stream = (
        spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format",               "csv")
             .option("cloudFiles.schemaLocation",       schema_location)
             .option("cloudFiles.inferColumnTypes",     "false")
             .option("cloudFiles.schemaEvolutionMode",  "rescue")
             .option("cloudFiles.includeExistingFiles", "true")
             .load(source_path)
             .filter(F.col("patient_id").isNotNull())            # quarantine: drop rows missing PK
             .withColumn("_source_file",       F.col("_metadata.file_path"))
             .withColumn("_source_file_mtime", F.col("_metadata.file_modification_time"))
             .withColumn("_load_datetime",     F.current_timestamp())
             .withColumn("_batch_id",          F.expr("uuid()"))
    )

    query = (
        stream.writeStream
              .format("delta")
              .outputMode("append")
              .option("checkpointLocation", checkpoint_path)
              .option("mergeSchema",        "true")
              .trigger(availableNow=True)
              .toTable(BRONZE_TABLE)
    )

    query.awaitTermination()

In [0]:
ingest_bronze()  # input_file_name replaced with _metadata.file_path in ingest_bronze's code

In [0]:
%sql

select * from rwd.bronze.encounter

Encounter_ID,Patient_ID,Cancer_Disease_Status,Last_Update,On_Treatment,_source_file,_source_file_mtime,_load_datetime,_batch_id,_rescued_data
E001,P001,Active,2026-01-15,Yes,/Volumes/rwd/bronze/myvolume/bronze/staging/encounter/encounter_raw.csv,2026-03-17T16:15:39.000Z,2026-03-17T16:48:42.911Z,e1db2a47-114d-4f37-ac45-f4997c065ae8,null
E002,P002,Remission,2025-12-20,No,/Volumes/rwd/bronze/myvolume/bronze/staging/encounter/encounter_raw.csv,2026-03-17T16:15:39.000Z,2026-03-17T16:48:42.911Z,11978013-1b4a-40d5-9af4-27bfbffc7cd5,null
E003,P003,Progressive,2026-01-10,Yes,/Volumes/rwd/bronze/myvolume/bronze/staging/encounter/encounter_raw.csv,2026-03-17T16:15:39.000Z,2026-03-17T16:48:42.911Z,76124a14-32b4-43b0-9881-67e01dc2196b,null
E004,P004,Stable,2026-01-05,Yes,/Volumes/rwd/bronze/myvolume/bronze/staging/encounter/encounter_raw.csv,2026-03-17T16:15:39.000Z,2026-03-17T16:48:42.911Z,fdf2a318-7da5-485f-a0e6-68dcfe53b4e1,null
E005,P005,Active,2026-01-18,Yes,/Volumes/rwd/bronze/myvolume/bronze/staging/encounter/encounter_raw.csv,2026-03-17T16:15:39.000Z,2026-03-17T16:48:42.911Z,d4ead0e5-69e1-4a7c-9598-71228f379076,null
E006,P006,Remission,2025-11-30,No,/Volumes/rwd/bronze/myvolume/bronze/staging/encounter/encounter_raw.csv,2026-03-17T16:15:39.000Z,2026-03-17T16:48:42.911Z,1008b92f-7283-47cb-9919-bb522de5901d,null
E007,P007,Progressive,2026-01-22,Yes,/Volumes/rwd/bronze/myvolume/bronze/staging/encounter/encounter_raw.csv,2026-03-17T16:15:39.000Z,2026-03-17T16:48:42.911Z,8c93076b-0978-427b-9d0d-51dcf1235e74,null
E008,P008,Stable,2026-01-12,Yes,/Volumes/rwd/bronze/myvolume/bronze/staging/encounter/encounter_raw.csv,2026-03-17T16:15:39.000Z,2026-03-17T16:48:42.911Z,b4723b6d-1ad8-4aa4-a187-3a17c131e78b,null
E009,P009,Active,2026-01-08,Yes,/Volumes/rwd/bronze/myvolume/bronze/staging/encounter/encounter_raw.csv,2026-03-17T16:15:39.000Z,2026-03-17T16:48:42.911Z,5d7e5329-9e97-4536-a51c-6b37b48f792f,null
E010,P010,Remission,2025-12-25,No,/Volumes/rwd/bronze/myvolume/bronze/staging/encounter/encounter_raw.csv,2026-03-17T16:15:39.000Z,2026-03-17T16:48:42.911Z,0754344e-6434-4b1a-be4d-f412f76d39cc,null
